In [17]:
import os
import json
# from pathlib import path
from openai import OpenAI
import pandas as pd
import re
import datetime as dt
import random
import shutil

from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple
import hashlib



In [ ]:

client = OpenAI(
    api_key=""
)

In [19]:
class JsonDiskCache:
    def __init__(self, cache_dir: str | Path):
        self.cache_dir = Path(f"./{cache_dir}/")
        self.cache_dir.mkdir(parents=True, exist_ok=True)

    def _path(self, key: str) -> Path:
        h = hashlib.sha256(key.encode("utf-8")).hexdigest()
        return self.cache_dir / f"{h}.json"

    def set(self, key: str, obj) -> None:
        self._path(key).write_text(json.dumps(obj, ensure_ascii=False), encoding="utf-8")

    def get(self, key: str):
        p = self._path(key)
        if not p.exists():
            return None
        data = json.loads(p.read_text(encoding="utf-8"))
        if isinstance(data, str):
            data = json.loads(data)
        return data

In [ ]:
def clean_and_parse_json(s: str):
    s = s.strip()

    # Strip ```json ... ``` fences if present
    if s.startswith("```"):
        s = re.sub(r"^```(?:json)?/s*", "", s, flags=re.IGNORECASE)
        s = re.sub(r"/s*```$", "", s)

    # If the model wrapped JSON in extra text, salvage the first {...} block
    start, end = s.find("{"), s.rfind("}")
    if start != -1 and end != -1 and end > start:
        s = s[start:end+1]

    data = json.loads(s)

    if isinstance(data, str):
        data = json.loads(data)

    return data

def extract_features_zeroshot(prompt, filepath):
    file = client.files.create(
        file=open(filepath, "rb"),
        purpose="user_data")
    completion = client.chat.completions.create(
        model="gpt-4.1",
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": prompt,
                    },                
                    {
                        "type": "file",
                        "file": {
                            "file_id": file.id,
                        }},]}])
    # Parse and return the response
    response = completion.choices[0].message.content.strip()
    # Remove markdown code blocks
    if response.startswith("```"):
        response = response.split("```")[1]
        if response.startswith("json"):
            response = response[4:]
        response = response.strip()
    if response.endswith("```"):
        response = response.rsplit("```", 1)[0].strip()
    
    # Remove escaped newlines and slashes
    response = response.replace("//n", "").replace("///", "/")
    
    cleaned_response = clean_and_parse_json(response)

    return cleaned_response

def zeroshot_batch(prompt, batch_path, cache):
    documents = os.listdir(batch_path)
    outputs = []
    for doc in documents:
        doc_path = batch_path+doc
        cache_key = f"search+detail::{doc_path}"
        cached = cache.get(cache_key)
        print(f"processing {doc_path}. saving as {cached}")
        # print(cached)
        if cached is not None:
            return cached
        else:
            output = extract_features_zeroshot(prompt, doc_path)
            # print(output)
            # cache.set(cache_key, str(output))
            cache.set(cache_key, json.dumps(output, ensure_ascii=False))
            outputs.append(output)

    return outputs

def process_zeroshot(prompt, batch_path="./evaluation_set/training/", note=""):
    now = dt.datetime.now()
    out_label = "{}_{}-{}{}".format(note, now.date(), now.hour, now.minute)
    out_path = Path("./outputs/zero-shot/" + out_label + "/")
    out_path.mkdir(parents=True, exist_ok=True)
    txt_filename = f"{out_label}.txt" 
    out_prompt_txt = out_path / txt_filename
    with open (out_prompt_txt, "w") as f:
        f.write(prompt)    
    cache = JsonDiskCache(out_path/Path(".cache"))
    feature_list = zeroshot_batch(prompt=prompt, batch_path=batch_path, cache=cache)

    outpath = out_path / f"{out_label}.csv"
    try:
        df = pd.DataFrame(feature_list)
        df.to_csv(outpath, encoding="utf-8-sig", index=False)
    except:
        print("Unable to render table")
    return feature_list


# Final combined prompt 

In [21]:
# Instructions string extracted from: /mnt/data/Copy of Instruction Prompt - Myneni.docx

instructions = """Complete Instructions for Data Extraction from Research Articles

CRITICAL VALIDATION RULES

BEFORE EXTRACTION: Verify the study meets inclusion criteria:

Include studies if they were published from 2019 to the present and written in English.

Include studies that evaluate an intervention with a digital health component targeting modifiable behavioral risk or protective factors for Alzheimer’s and related diseases (ADRD)

Include individuals not yet diagnosed with ADRD.

Studies are eligible if they assess feasibility (e.g., recruitment or data collection approaches), acceptability (e.g., participant evaluation of intervention content), psychosocial outcomes (e.g., changes in attitudes or self-efficacy toward a behavior), behavioral outcomes (e.g., changes in actual behavioral performance), or health outcomes

Exclude studies if they were published before 2019 or written in a language other than English.

Exclude studies if participants included individuals diagnosed with ADRD or a related diagnosis, or if participants have amnestic cognitive impairment (representing atypically severe decline for age).

Interventions evaluating diagnostic or pharmaceutical to ADRD prevention approaches rather than behavioral interventions are excluded.

Studies that do not evaluate interventions are excluded, as are cross-sectional studies, cohort studies, meta-analyses, reviews, systematic reviews.

Studies that do not collect or analyze empirical data (e.g., protocols, studies using simulated data) are excluded.

Studies for which the intervention requires clinic-bound technology or equipment, like transcranial stimulation, were excluded.

EXTRACTION PRINCIPLE: Only extract information explicitly stated in the provided text

Never infer, assume, or add information not directly present

If information is missing, write "N/A"

Quote exact terminology from the paper

REQUIRED DATA EXTRACTION FIELDS

1. Author Year:

Extract the author names and year.

2. Target Population :  Extract only what is explicitly stated. For example, “Older adults” or “Patients with MCI”

Group of people on which the study is being performed, for whom the intervention was designed, and on whom the intervention is being evaluated.

3. Inclusion and Exclusion Criteria:

Extract the requirements participants must meet to participate in the study.

4. Sample Size:

Number of participants that participated in the study.

5. Mean Age and Standard Deviation:

Mean age of participant and standard deviation.

6. Study Design:

Extract the scientific approach that was used to evaluate the intervention.

Use exact methodology terminology from the paper.

7. Intervention Description: Include only details explicitly provided. Do not use vague statements like "Specific protocols, dosing, duration, and frequency varied across the included studies." Summarize or be detailed when possible

Briefly describe the intervention and any its features that are thought to influence ADRD risk among participants.

Protocols, dosing, duration, frequency

Delivery methods and personnel

Session length and total number of sessions

Additional components or comparison conditions

8. Technology:

Report the digital or computerized components of the interventions that were used and briefly describe them using the exact terminology used in the paper.

9. Theoretical Basis

If the intervention was designed following a theoretical health promotion framework, state the framework using the exact terminology from the paper.

10. Outcome Variables

List the exact outcome variables and outcome measure as defined by the authors.

11. Results

Must Include:

Statistical results for the target population only

The changes in the outcome variables by study group (if applicable), including effect size and whether it was statistically significant or not.

Assessment timeline (baseline, post-intervention, follow-up periods)

QUALITY CONTROL CHECKLIST

Before submitting the extraction, verify that:

The study includes patients that do not have Alzheimer's and related dementia.

Each data point appears exactly as stated in the source text.

The extraction uses the exact terminology and quotes from the paper.

Missing information is marked N/A.

All required fields are completed.

Quality Check: Before submitting, re-read each extracted data point and verify that it appears in the source text exactly as stated. An example of data extracted is below. Please format results in the table structure provided and verify each data point against the source text before finalizing.

EXAMPLE OUTPUT FORMAT

Embon-Magal (2022)\tOlder adults\tOlder adults > 65 who participate in leisure activities on a daily basis at the adult day center. Ability to walk independently with or without an assistive device. Ability to understand simple instructions and sign an informed consent
consent form. Ability to commit to the program based on a short interview.\t47\t81.16 (8.23)\tA single-blind, randomized, and controlled clinical trial\tTIM is a combined motor-cognitive co-dependent intervention inspired by the Eshkol-Wachman Movement Notation (EWMN), which uses graphic symbols to describe motion. Unlike the fixed use of symbols in EWMN, TIM departs from the permanence system of symbols and uses the ephemerality of symbols and frequent changes in them as a central tool for cognitive simulation\tThinking In Motion (TIM), a co-dependent motor-cognitive intervention, and CogniFit, a single modality computerized cognitive training.\tN/A\tN/A\tPrimary outcome: cognitive functions. Secondary outcome: gait performance.\tCogniFit total Z scores significantly improved from baseline to post-intervention for both groups. Single-modality (CogniFit) and co-dependent (TIM) trainings improve cognition but not gait in older adults with cognitive impairment. Analyses of gait speed under ST condition revealed no main effect for time or group, and no significant time X group interaction.
"""

# Optional quick check
print(instructions)
print("\n---\nChars:", len(instructions))


Complete Instructions for Data Extraction from Research Articles

CRITICAL VALIDATION RULES

BEFORE EXTRACTION: Verify the study meets inclusion criteria:

Include studies if they were published from 2019 to the present and written in English.

Include studies that evaluate an intervention with a digital health component targeting modifiable behavioral risk or protective factors for Alzheimer’s and related diseases (ADRD)

Include individuals not yet diagnosed with ADRD.

Studies are eligible if they assess feasibility (e.g., recruitment or data collection approaches), acceptability (e.g., participant evaluation of intervention content), psychosocial outcomes (e.g., changes in attitudes or self-efficacy toward a behavior), behavioral outcomes (e.g., changes in actual behavioral performance), or health outcomes

Exclude studies if they were published before 2019 or written in a language other than English.

Exclude studies if participants included individuals diagnosed with ADRD or a re

#### Not enforcing json output structure or providing example pairs

In [23]:
cache = JsonDiskCache("./.cache_openai_fewshot_combined")




In [ ]:
def build_schema_from_model_json(model_json: Dict[str, Any]) -> Dict[str, Any]:
    props = {k: {"type": "string"} for k in model_json.keys()}

    item_object_schema = {
        "type": "object",
        "additionalProperties": False,
        "properties": props,
        "required": list(model_json.keys()),
    }

    return {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "items": {
                "type": "array",
                "minItems": 1,
                "maxItems": 1,
                "items": item_object_schema,
            }
        },
        "required": ["items"],
    }


In [ ]:
import json
import hashlib
from pathlib import Path
from typing import Any, Dict, List, Optional

def sha256_file(path: str | Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def upload_cached_file_id(client, path: str | Path, cache) -> str:
    path = Path(path)
    key = f"file_id::{sha256_file(path)}::{path.name}"
    cached = cache.get(key)
    if cached:
        return cached["file_id"]

    with open(path, "rb") as f:
        uploaded = client.files.create(file=f, purpose="user_data")
    cache.set(key, {"file_id": uploaded.id})
    return uploaded.id


In [ ]:
def responses_myneni_api(
    client,
    filepath: str | Path,
    *,
    instructions: str,
    cache,
    model: str = "gpt-4.1",
    model_json
) -> List[Dict[str, Any]]:

    schema = build_schema_from_model_json(model_json)
    target_id = upload_cached_file_id(client, filepath, cache)

    input_messages: List[Dict[str, Any]] = []

    
    input_messages.append(
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": "Extract from attached PDF per system instructions."},
                {"type": "input_file", "file_id": target_id},
            ],
        }
    )

    resp = client.responses.create(
        model=model,
        temperature=0,
        instructions=instructions,
        input=input_messages,
        text={
            "format": {
                "type": "json_schema",
                "name": "study_features",
                "strict": True,
                "schema": schema,
            }
        }
    )

    raw = resp.output_text
    
    parsed = json.loads(raw)
    if parsed["items"] and isinstance(parsed["items"], list) and len(parsed["items"]) > 0:
        parsed["items"][0]["response_id"] = resp.id
    return parsed['items']


In [27]:
print(instructions)

Complete Instructions for Data Extraction from Research Articles

CRITICAL VALIDATION RULES

BEFORE EXTRACTION: Verify the study meets inclusion criteria:

Include studies if they were published from 2019 to the present and written in English.

Include studies that evaluate an intervention with a digital health component targeting modifiable behavioral risk or protective factors for Alzheimer’s and related diseases (ADRD)

Include individuals not yet diagnosed with ADRD.

Studies are eligible if they assess feasibility (e.g., recruitment or data collection approaches), acceptability (e.g., participant evaluation of intervention content), psychosocial outcomes (e.g., changes in attitudes or self-efficacy toward a behavior), behavioral outcomes (e.g., changes in actual behavioral performance), or health outcomes

Exclude studies if they were published before 2019 or written in a language other than English.

Exclude studies if participants included individuals diagnosed with ADRD or a re

In [28]:
model_json = {
    "author_year": "",
    "target_population": "",
    "inclusion_criteria": "",
    "sample_size": "",
    "mean_age_sd": "",
    "study_design": "",
    "intervention_description": "",
    "technology_used": "",
    "theoretical_basis": "",
    "outcome_variables": "",
    "results": "",
    "include": ""
}

In [38]:
myneni_output = responses_myneni_api(
    client=client, 
    filepath = "C:/Users/Rodrigo/OneDrive/ADRD Scoping Review/Screener/Full_Extraction/evaluation_set/training/Acevedo et al. - 2022 - Cognitive Training with Neurofeedback Using fNIRS .pdf", 
    instructions = instructions, 
    cache = cache, 
    model = "gpt-4.1",
    model_json=model_json
    )

In [39]:
myneni_output

[{'author_year': 'Acevedo et al., 2022',
  'target_population': 'Older adults (healthy adults, ages 54–84)',
  'inclusion_criteria': 'N/A (Eligibility confirmed via online screening; participants were healthy adults, not diagnosed with ADRD or cognitive impairment)',
  'sample_size': '86 (ABC: n = 50; Tetris: n = 36)',
  'mean_age_sd': 'ABC: 68.56 (6.80); Tetris: 63.36 (5.61)',
  'study_design': 'Randomized controlled trial',
  'intervention_description': '4-week cognitive training program with neurofeedback (CT-NF). Treatment group (ABC) played three different app-based cognitive games (Random Numbers, Endless Addition, Forwards Order) three times daily, about 10–15 min per day, 5–7 days per week, while wearing a brain sensor. Control group played three rounds of Tetris daily, similar duration and frequency, also with brain sensor. Both groups received live, virtual training and were instructed to log brain activity and game scores after each session.',
  'technology_used': 'XB-01 fun

In [40]:
def responses_process_folder(instructions, folder_path, note=""):
    print(instructions)
    outlist=[]
    now = dt.datetime.now()
    out_label = "{}_{}-{}{}".format(note, now.date(), now.hour, now.minute)
    out_path = Path("./outputs/few-shot/" + out_label + "/")
    out_path.mkdir(parents=True, exist_ok=True)
    txt_filename = f"{out_label}.txt" 
    out_prompt_txt = out_path / txt_filename
    with open (out_prompt_txt, "w") as f:
        f.write(instructions)    
    for fil in os.listdir(folder_path):
        filepath = folder_path+fil
        print(f"\nProcessing {fil}")
        out = responses_myneni_api(
            client,
            filepath=filepath,
            instructions=instructions,
            cache=cache,
            model="gpt-4.1",
            model_json=model_json
)  # choose a model that supports PDF input :contentReference[oaicite:1]{index=1}
        outlist.append(out[0])
        # print(out[0])
    csv_filename = f"{out_label}.csv" 
    out_csv = out_path / csv_filename
    # out_fixed = [out[0] for out in outlist]
    pd.DataFrame(outlist).to_csv(out_csv)
    return outlist


In [33]:
instructions

'Complete Instructions for Data Extraction from Research Articles\n\nCRITICAL VALIDATION RULES\n\nBEFORE EXTRACTION: Verify the study meets inclusion criteria:\n\nInclude studies if they were published from 2019 to the present and written in English.\n\nInclude studies that evaluate an intervention with a digital health component targeting modifiable behavioral risk or protective factors for Alzheimer’s and related diseases (ADRD)\n\nInclude individuals not yet diagnosed with ADRD.\n\nStudies are eligible if they assess feasibility (e.g., recruitment or data collection approaches), acceptability (e.g., participant evaluation of intervention content), psychosocial outcomes (e.g., changes in attitudes or self-efficacy toward a behavior), behavioral outcomes (e.g., changes in actual behavioral performance), or health outcomes\n\nExclude studies if they were published before 2019 or written in a language other than English.\n\nExclude studies if participants included individuals diagnosed 

In [41]:
outlist = responses_process_folder(instructions=instructions, folder_path="C:/Users/Rodrigo/OneDrive/ADRD Scoping Review/Screener/Full_Extraction/evaluation_set/training/", note = "myneni_prompt")

Complete Instructions for Data Extraction from Research Articles

CRITICAL VALIDATION RULES

BEFORE EXTRACTION: Verify the study meets inclusion criteria:

Include studies if they were published from 2019 to the present and written in English.

Include studies that evaluate an intervention with a digital health component targeting modifiable behavioral risk or protective factors for Alzheimer’s and related diseases (ADRD)

Include individuals not yet diagnosed with ADRD.

Studies are eligible if they assess feasibility (e.g., recruitment or data collection approaches), acceptability (e.g., participant evaluation of intervention content), psychosocial outcomes (e.g., changes in attitudes or self-efficacy toward a behavior), behavioral outcomes (e.g., changes in actual behavioral performance), or health outcomes

Exclude studies if they were published before 2019 or written in a language other than English.

Exclude studies if participants included individuals diagnosed with ADRD or a re

### Confirm few-shot
